# L-shape magnetostatics with the two quantum Jacobi circuits

The L-shaped object ($\mu_r = 10$) on an $8^3$ grid, $A_z = 10^{-2}$ at $x=0$, $A_z = 5\cdot10^{-2}$ at $x=1$, $\partial_y A_z = 0$ at $y=0,1$, $J = 0$. Each damped Jacobi step is

$$a^{\text{new}} = \beta\,\frac{S - J}{\Delta} + (1-\beta)\,a .$$

The step is computed three ways:

| Method | Circuit | Qubits | Used here for |
|---|---|---|---|
| classical | blocks + slices | — | reference |
| **all frames** (`test_jacobi_all_frames_weighted.py`) | each frame encoded in its own layout, frame register, star shifter, constants β and 1−β | 23 | **full solve** to convergence |
| **ctrl-π opt** (`test_jacobi_ctrl_pi_opt.py`, here at q = 3) | 12 products + J + a·Δ encoded **once**, ctrl-π, structured MAP, star shifter, W1 = 1/Δ, W2 | 28 | the **first 5 steps** (≈2–3 min per step), compared step by step |

Then we compute $\mathbf B = \nabla\times\mathbf A$ and plot $A$ and $B$ on the mid z-plane.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit import Gate
from qiskit.circuit.library import UCRYGate
from qiskit_aer import AerSimulator

import test_combined_frames_aer as C              # L-shape (mu_r = 10), faces, blocks, layouts (q = 3)
import test_jacobi_all_frames_weighted as AW      # all-frames circuit
import test_jacobi_ctrl_pi_opt as O               # structured labels (ROLE, label, KF, w2 constants)
from star_shifter import build_proposed
from test_jacobi_ctrl_pi import save_field_txt
from circuit_saving import save_circuits

q, n, beta, FRAMES = C.q, C.n, C.beta, C.FRAMES
h = 1.0/(n - 1)
K, I, J_ = C.K, C.I, C.J_
print(f"grid {n}^3, mu_r = {C.mu_r}, beta = {beta}")

## 1. Solver loop (shared by all methods)

In [ ]:
def apply_bc_az(az, A_left=1e-2, A_right=5e-2):
    az[:, 0, :] = A_left; az[:, -1, :] = A_right
    az[:, 1:-1, 0] = az[:, 1:-1, 1]; az[:, 1:-1, -1] = az[:, 1:-1, -2]
    return az

J0 = {f: np.zeros((n-1, n, n)) for f in FRAMES}          # J = 0 in the L-shape problem

def run(step, nsteps=2000, conv_tol=1e-7, keep=False, label=""):
    ax = np.zeros((n, n-1, n)); ay = np.zeros((n, n, n-1)); az = apply_bc_az(np.zeros((n-1, n, n)))
    hist = [(ax, ay, az)] if keep else None
    t0 = time.time()
    for t in range(nsteps):
        new = step(ax, ay, az, J0)                          # interior values, frame layout
        full = {}
        for f in FRAMES:
            arr = np.zeros((n-1, n, n)); arr[K, I, J_] = new[f]; full[f] = arr
        axn, ayn, azn = C.xyz_to_zxy(full['x']), C.yzx_to_zxy(full['y']), apply_bc_az(full['z'])
        err = np.linalg.norm(azn - az)**2
        ax, ay, az = axn, ayn, azn
        if keep: hist.append((ax, ay, az))
        if label: print(f"  {label} step {t+1}: |dAz|^2 = {err:.3e}  ({time.time()-t0:.0f} s)", flush=True)
        if err < conv_tol:
            break
    return (ax, ay, az), t + 1, hist

## 2. Classical reference and the all-frames circuit (full solve)

In [ ]:
(axc, ayc, azc), nc, hist_c = run(AW.step_classical, keep=True)
print(f"classical      : converged after {nc} steps")

t0 = time.time()
(axq, ayq, azq), nq, _ = run(AW.step_quantum)
print(f"all-frames (23 qubits): converged after {nq} steps, {time.time()-t0:.0f} s, {nq} circuit runs")
print(f"max |A_quantum - A_classical|:  ax {np.abs(axq-axc).max():.1e}, ay {np.abs(ayq-ayc).max():.1e}, "
      f"az {np.abs(azq-azc).max():.1e}")

## 3. The ctrl-π opt circuit at q = 3 (28 qubits)

This is the same construction as `test_jacobi_ctrl_pi_opt.py`, rebuilt for the $8^3$ grid:

- the 12 edge–face products, $J$ and $a\cdot\Delta$ are encoded **once**, in the storage layout, with the structured label $L = s + 2h + 4d + 16\,\text{kind}$;
- FRAME, then COPY, then ctrl-π;
- MAP-A ($d\to r$), MAP-B, STAR SHIFTER†, UNMAP-B, W2, UNMAP-A;
- W1 ($1/\Delta$ at the target), then SUM.

In [ ]:
sh, _, _ = build_proposed(q)
NQS = sh.num_qubits                                          # 18
IDX = list(range(3*q)); TQ = [3*q + t for t in range(4)]
G = [list(range(0, q)), list(range(q, 2*q)), list(range(2*q, 3*q))]
LQ = list(range(NQS, NQS + 5)); FQ = [NQS + 5, NQS + 6]; CQ = NQS + 7; W1 = NQS + 8; W2 = NQS + 9
NQP = NQS + 10                                               # 28
S_, H_, D0, D1, KIND = LQ

def frame_prep(c):
    c.ry(2*np.arccos(np.sqrt(2/3)), FQ[1]); c.x(FQ[1]); c.ch(FQ[1], FQ[0]); c.x(FQ[1])

def ctrl_pi(c):
    for ctrl, ga, gb in [(FQ[0], G[0], G[2]), (FQ[0], G[1], G[2]), (FQ[1], G[0], G[1]), (FQ[1], G[1], G[2])]:
        for a_, b_ in zip(ga, gb):
            c.cswap(ctrl, a_, b_)

def mod3_step(c, frame_state, up):
    s01 = lambda: c.mcx(FQ + [D1], D0, ctrl_state=frame_state)
    s02 = lambda: c.mcx(FQ + [D0], D1, ctrl_state=frame_state)
    (s01(), s02()) if up else (s02(), s01())

def map_a(c, inverse=False):
    mod3_step(c, 0b00, up=not inverse); mod3_step(c, 0b01, up=inverse)

def map_b(c):
    al, sg, r1, r0 = TQ
    c.mcx([KIND, D0, D1], r0, ctrl_state=0b000)
    c.mcx([KIND, D0, S_], r1, ctrl_state=0b110); c.mcx([KIND, D1, S_], r1, ctrl_state=0b110)
    c.mcx([KIND, D0, D1, H_], al, ctrl_state=0b0000); c.mcx([KIND, D1], al, ctrl_state=0b10)
    c.mcx([KIND, D0, D1, S_], sg, ctrl_state=0b0000)
    c.mcx([KIND, D0, CQ], sg, ctrl_state=0b110); c.mcx([KIND, D1, CQ], sg, ctrl_state=0b110)

# material is fixed -> 1/D lookup (W1) is fixed
invD = {}
for f in FRAMES:
    D = np.zeros((n, n, n)); D[K, I, J_] = C.diag_D(*C.faces[f])
    invD[f] = np.where(D > 0, 1/np.where(D > 0, D, 1), 0.0)
invD_max = max(v.max() for v in invD.values())
w1 = np.zeros(4*n**3)
for fi, f in enumerate(FRAMES):
    w1[fi*n**3:(fi + 1)*n**3] = C.to_flat(invD[f])/invD_max

def build_body(regs=None, barriers=False):
    b = QuantumCircuit(*regs) if regs else QuantumCircuit(NQP)
    bar = (lambda lab: b.barrier(label=lab)) if barriers else (lambda lab: None)
    bar("FRAME, COPY"); frame_prep(b); b.h(CQ)
    bar("ctrl-π"); ctrl_pi(b)
    bar("MAP-A d→r"); map_a(b)
    bar("MAP-B"); map_b(b)
    bar("STAR SHIFTER†"); b.compose(sh.inverse(), qubits=list(range(NQS)), inplace=True)
    bar("UNMAP-B"); map_b(b)
    bar("W2: β, 1−β, signs"); b.append(UCRYGate(list(O.w2_angles())), [W2, S_, H_, D0, D1, KIND, CQ])
    bar("UNMAP-A r→d"); map_a(b, inverse=True)
    bar("W1: 1/Δ(p)"); b.append(UCRYGate(list(2*np.arccos(w1))), [W1] + IDX + FQ)
    bar("SUM"); b.h(LQ + [CQ])
    return b

sim = AerSimulator(method="statevector")
body_t = transpile(build_body(), sim, optimization_level=0)
print(f"ctrl-π opt circuit: {NQP} qubits")

def encode_opt(ax, ay, az, Js):
    fr = C.frames_of(ax, ay, az); blk = {}
    for f in FRAMES:
        Q = C.blocks(*fr[f], *C.faces[f])
        for nm, (r, normal, s) in O.ROLE.items():
            hh = (normal - r - 1) % 3; d = (r + O.KF[f]) % 3
            arr = C.frame_to_storage[f](C.pad(Q[nm])); L = O.label(0, d, hh, s)
            if L in blk:
                assert np.allclose(blk[L], arr)
            blk[L] = arr
        D = np.zeros((n, n, n)); D[K, I, J_] = C.diag_D(*C.faces[f])
        blk[O.label(1, O.KF[f], 0, 0)] = C.frame_to_storage[f](C.pad(Js[f]))
        blk[O.label(1, O.KF[f], 0, 1)] = C.frame_to_storage[f](C.pad(fr[f][0])*D)
    enc = np.zeros((32, n**3))
    for L, arr in blk.items():
        enc[L] = C.to_flat(arr)
    return enc, np.linalg.norm(enc)

def step_ctrl_pi(ax, ay, az, Js):
    enc, norm = encode_opt(ax, ay, az, Js)
    psi0 = np.zeros(2**NQP, complex)
    for L in range(32):
        psi0[L*2**LQ[0]: L*2**LQ[0] + n**3] = enc[L]/norm
    qc = QuantumCircuit(NQP); qc.set_statevector(psi0); qc.compose(body_t, inplace=True); qc.save_statevector()
    sv = sim.run(qc).result().get_statevector().data
    out = {}
    for fi, f in enumerate(FRAMES):
        ro = sv[fi*2**FQ[0]: fi*2**FQ[0] + n**3]
        out[f] = (C.from_flat(ro).real*8*np.sqrt(6)*norm*invD_max)[K, I, J_]
    return out

### First 5 Jacobi steps with ctrl-π opt, compared with the classical iterates

In [ ]:
N_PI = 5
(axp, ayp, azp), _, hist_p = run(step_ctrl_pi, nsteps=N_PI, conv_tol=-1, keep=True, label="ctrl-π opt")
print("\nstep   max|A_ctrl-π - A_classical| (ax, ay, az)")
for s in range(1, N_PI + 1):
    (a1, a2, a3), (b1, b2, b3) = hist_p[s], hist_c[s]
    print(f"{s:4d}   {np.abs(a1-b1).max():.1e}  {np.abs(a2-b2).max():.1e}  {np.abs(a3-b3).max():.1e}")

## 4. Magnetic flux density $\mathbf B = \nabla\times\mathbf A$

$\mathbf B$ lives on the faces. With arrays indexed $[z, x, y]$:

$$
B_x = \partial_y A_z - \partial_z A_y,\quad
B_y = \partial_z A_x - \partial_x A_z,\quad
B_z = \partial_x A_y - \partial_y A_x .
$$

On the mid z-plane, $B_x$ and $B_y$ are averaged to cell centres for plotting.

In [ ]:
def curl(ax, ay, az):
    Bx = ((az[:, :, 1:] - az[:, :, :-1]) - (ay[1:, :, :] - ay[:-1, :, :]))/h     # (n-1, n,   n-1)
    By = ((ax[1:, :, :] - ax[:-1, :, :]) - (az[:, 1:, :] - az[:, :-1, :]))/h     # (n-1, n-1, n  )
    Bz = ((ay[:, 1:, :] - ay[:, :-1, :]) - (ax[:, :, 1:] - ax[:, :, :-1]))/h     # (n,   n-1, n-1)
    return Bx, By, Bz

def midplane(ax, ay, az):
    kz = (n - 1)//2
    Bx, By, Bz = curl(ax, ay, az)
    bx = 0.5*(Bx[kz, :-1, :] + Bx[kz, 1:, :])        # cell centres (n-1, n-1)
    by = 0.5*(By[kz, :, :-1] + By[kz, :, 1:])
    return az[kz], bx, by, kz

sols = {"classical (converged)": (axc, ayc, azc),
        "all-frames circuit (converged)": (axq, ayq, azq),
        f"classical, step {N_PI}": hist_c[N_PI],
        f"ctrl-π opt circuit, step {N_PI}": hist_p[N_PI]}
for name, (a1, a2, a3) in sols.items():
    _, bx, by, _ = midplane(a1, a2, a3)
    print(f"{name:34s}: max |B| on mid z-plane = {np.hypot(bx, by).max():.4f}")

## 5. Plots: $A_z$ and $\mathbf B$ on the mid z-plane

In [ ]:
xn = np.linspace(0, 1, n); xc = (np.arange(n - 1) + 0.5)*h
L_mid = C.L_mask[:, :, (n - 1)//2]

fig, axs = plt.subplots(2, 4, figsize=(20, 9.5), constrained_layout=True)
for col, (name, (a1, a2, a3)) in enumerate(sols.items()):
    azm, bx, by, kz = midplane(a1, a2, a3)
    cf = axs[0, col].contourf(xn, xn, azm.T, levels=20, cmap="viridis")
    axs[0, col].contour(xc, xc, L_mid.T.astype(float), levels=[0.5], colors="w", linewidths=2)
    axs[0, col].set_title(f"$A_z$ — {name}")
    fig.colorbar(cf, ax=axs[0, col], shrink=0.8)
    pm = axs[1, col].pcolormesh(np.linspace(0, 1, n), np.linspace(0, 1, n), np.hypot(bx, by).T, cmap="magma")
    axs[1, col].quiver(xc, xc, bx.T, by.T, color="w", scale_units="xy")
    axs[1, col].contour(xc, xc, L_mid.T.astype(float), levels=[0.5], colors="c", linewidths=2)
    axs[1, col].set_title(f"$|B|$ and $(B_x, B_y)$ — {name}")
    fig.colorbar(pm, ax=axs[1, col], shrink=0.8)
for a in axs.ravel():
    a.set_aspect("equal"); a.set_xlabel("x"); a.set_ylabel("y")
fig.suptitle(f"L-shape, mid z-plane (k = {kz}), μ_r = {C.mu_r}")
import os; os.makedirs("results/L_shape_two_methods", exist_ok=True)
fig.savefig("results/L_shape_two_methods/A_B_midplane.svg", format="svg")
plt.show()

## 6. Save results and circuits

In [ ]:
import os
res = "results/L_shape_two_methods"; os.makedirs(res, exist_ok=True)
to_save = {"classical_converged": (axc, ayc, azc), "allframes_converged": (axq, ayq, azq),
           f"classical_step{N_PI}": hist_c[N_PI], f"ctrlpi_step{N_PI}": hist_p[N_PI]}
for tag, (a1, a2, a3) in to_save.items():
    for nm, arr in (("ax", a1), ("ay", a2), ("az", a3)):
        save_field_txt(f"{res}/{nm}_{tag}.txt", arr, f"{nm}, L-shape, {tag}")
print(f"saved ax/ay/az for {list(to_save)} in {res}/  (shape-preserving, one block per k)")

# circuits (SVG): the all-frames circuit and the ctrl-π opt circuit at q = 3
regs = list(sh.qregs) + [QuantumRegister(5, "L"), QuantumRegister(2, "frame"), QuantumRegister(1, "copy"),
                         QuantumRegister(1, "w1"), QuantumRegister(1, "w2")]
det = QuantumCircuit(*regs, name="ctrl_pi_opt_q3")
det.append(Gate("ENCODE products, J, a·Δ", len(IDX + LQ), []), IDX + LQ)
det.compose(build_body(regs=regs, barriers=True), inplace=True)
circs = {"ctrl_pi_opt_q3_gates": (det, {"fold": 60, "scale": 0.6})}
circs.update(AW.named_circuits())
save_circuits(circs, "circuits/L_shape_two_methods")